# [V3] Huan luyen RT-DETR-L - Zalo AI Traffic Sign 2020

Ban V3 nang cap tu ban goc voi 3 thay doi chinh:

| Hang muc | Ban cu | Ban V3 |
|---|---|---|
| Chia du lieu | 80% Train / 20% Val (khong co tap Test) | **70 / 10 / 20** co tap Hold-out Test rieng |
| So epoch | 50 | **100 + Early Stopping `patience=15`** |
| Nhat ky | Chi co `results.csv` tho | Xuat them `rtdetr_training_history.json` |

**Moi truong:** Google Colab (GPU T4). Chon notebook nay cho Colab vi RT-DETR chay o `imgsz=640` nen nhe hon, hop voi han muc GPU cua Colab; con YOLOv8 chay `imgsz=1280` thi nen de ben Kaggle.

**Chuan bi truoc:** co thong tin dang nhap Kaggle de tai dataset. Kaggle da doi sang token moi (chuoi `KGAT_...`), nen cach nhanh nhat la copy chuoi do vao Colab Secrets voi ten `KAGGLE_API_TOKEN`. Neu van muon dung file `kaggle.json` cu thi vao `kaggle.com/settings/api`, muc **Legacy API Credentials**, bam **Create Legacy API Key**. Cell 2 ho tro ca hai cach.

> ### Nguyen tac vang cua phien ban V3
>
> Toan bo du lieu duoc chia lai theo ty le **70% Train / 10% Validation / 20% Hold-out Test** bang `split_dataset.py` voi `random_seed=42`.
>
> - Thu muc `holdout_test/` **khong duoc khai bao trong `data.yaml`** va tuyet doi khong dung o bat ky buoc nao trong notebook nay.
> - Ca 3 model deu goi cung mot script chia, cung mot seed, nen chac chan dung chung mot tap Test.
> - Chi mo tap Hold-out ra dung mot lan duy nhat o notebook `evaluate_3_models.ipynb`.

## Cell 1: Ket noi Google Drive

Colab hay ngat phien bat ngo, nen phai ghi thang trong so vao Drive de khong mat cong train lai tu dau.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/DoAn_NhanDienBienBao'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Thu muc sao luu an toan: {SAVE_DIR}")

## Cell 2: Tai dataset tu Kaggle ve Colab

In [ ]:
!pip install -q kaggle

import os
import shutil

THU_MUC_KAGGLE = '/root/.kaggle'
os.makedirs(THU_MUC_KAGGLE, exist_ok=True)

# Kaggle hien co 2 kieu token, notebook nay ho tro ca hai:
#   - Token moi : mot chuoi bat dau bang KGAT_..., luu vao ~/.kaggle/access_token
#   - Token cu  : file kaggle.json tai ve, luu vao ~/.kaggle/kaggle.json
# Cach an toan nhat la de token trong Colab Secrets (bieu tuong chia khoa o thanh trai),
# dat ten la KAGGLE_API_TOKEN. Lam vay token khong bi luu vao notebook.

api_token = ''
try:
    from google.colab import userdata
    api_token = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    # Khong co Colab Secrets thi bo qua, xuong duoi thu cach khac
    api_token = ''

if api_token:
    with open(os.path.join(THU_MUC_KAGGLE, 'access_token'), 'w') as f:
        f.write(api_token.strip())
    os.chmod(os.path.join(THU_MUC_KAGGLE, 'access_token'), 0o600)
    print("Da cau hinh token moi (doc tu Colab Secrets).")

elif os.path.exists('kaggle.json'):
    shutil.copy('kaggle.json', os.path.join(THU_MUC_KAGGLE, 'kaggle.json'))
    os.chmod(os.path.join(THU_MUC_KAGGLE, 'kaggle.json'), 0o600)
    print("Da cau hinh token cu (file kaggle.json).")

else:
    raise FileNotFoundError(
        "Chua co thong tin dang nhap Kaggle. Chon 1 trong 2 cach:\n\n"
        "CACH 1 (khuyen dung) - Token moi qua Colab Secrets:\n"
        "  1. Vao kaggle.com/settings/api, bam tao API token, copy chuoi KGAT_...\n"
        "  2. Trong Colab, mo bieu tuong chia khoa o thanh trai, bam 'Add new secret'\n"
        "  3. Name = KAGGLE_API_TOKEN, Value = chuoi vua copy, bat cong tac Notebook access\n"
        "  4. Chay lai cell nay\n\n"
        "CACH 2 - Token cu:\n"
        "  1. Vao kaggle.com/settings/api, muc 'Legacy API Credentials',\n"
        "     bam 'Create Legacy API Key' de tai file kaggle.json\n"
        "  2. Upload file do vao thu muc lam viec cua Colab roi chay lai cell nay"
    )

In [ ]:
!kaggle datasets download -d phhasian0710/za-traffic-2020
!unzip -q -n za-traffic-2020.zip -d /content/dataset
print("Da tai va giai nen dataset.")

## Cell 3: Lay script chia du lieu dung chung

Dung dung file `split_dataset.py` ma YOLOv8 va Faster R-CNN dang dung. Seed 42 co dinh nen du chay ben Colab hay ben Kaggle thi phep chia van ra ket qua y het.

In [ ]:
import os
import urllib.request

# Keo file split_dataset.py tu GitHub ve de dung chung mot phep chia voi 2 model kia.
URL_SCRIPT = ('https://raw.githubusercontent.com/vtdung23/Object-Detection-Application/main/Traffic-Sign-Detection-ZaloAI/data_preparation/split_dataset.py')

if not os.path.exists('split_dataset.py'):
    try:
        urllib.request.urlretrieve(URL_SCRIPT, 'split_dataset.py')
        print("Da tai split_dataset.py tu GitHub ve.")
    except Exception as loi:
        raise FileNotFoundError(
            f"Khong tai duoc split_dataset.py ({loi}).\n"
            "Cach khac: mo file split_dataset.py trong repo, copy noi dung roi tao thu cong "
            "file cung ten o thu muc lam viec hien tai."
        )
else:
    print("File split_dataset.py da co san.")

In [ ]:
!python split_dataset.py --output-root /content/data_v3

DATA_YAML = '/content/data_v3/dataset_train_val/data.yaml'
print("\nNoi dung file data.yaml se dung de train:")
print(open(DATA_YAML, encoding='utf-8').read())

## Cell 4: Cai dat Ultralytics

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## Cell 5: Huan luyen RT-DETR-L (100 epochs + Early Stopping)

Giu nguyen `imgsz=640` va `batch=4` theo dung spec muc 3.2. Nhac lai ly do de `640`: ma tran Self-Attention phinh theo $O(N^2)$ nen anh `1280` vua gay tran VRAM vua keo dai thoi gian train qua khuon kho do an.

Early Stopping o day dac biet quan trong: Transformer noi tieng la **hoi tu cham hon CNN** o giai doan dau (vi phai tu hoc quan he khong gian tu con so 0 thay vi duoc cai san quy nap cuc bo nhu mang tich chap). Cho no tran 100 epoch va de `patience=15` tu quyet dinh diem dung se cong bang hon nhieu so voi viec cat cung o 50 epoch.

In [ ]:
from ultralytics import RTDETR

NUM_EPOCHS = 100
PATIENCE = 15

# Luon train tu checkpoint goc rtdetr-l.pt, khong resume tu checkpoint bi NaN
model = RTDETR('rtdetr-l.pt')

results = model.train(
    data=DATA_YAML,          # chi tro toi train + val
    epochs=NUM_EPOCHS,
    patience=PATIENCE,       # [V3] Early Stopping
    imgsz=640,               # Spec muc 3.2 - da ha tu 1280 de chong OOM
    batch=4,                 # Colab 1 GPU T4 (giam xg 2 neu OOM)
    optimizer='AdamW',
    lr0=1e-4,                # [FIX] Giam LR de on dinh Transformer
    amp=False,               # [FIX] Tat FP16 de tranh loi nan (Overflow)
    cos_lr=True,             # Cosine Annealing
    project=f'{SAVE_DIR}/zalo_traffic',
    name='rtdetr_v3',
    device=0,
)

## Cell 6: Ham xuat nhat ky huan luyen ra JSON

Dung chung ham voi notebook YOLOv8. Ham do cot loss theo tu khoa nen tu xu ly duoc chuyen RT-DETR dung bo loss khac (`giou_loss`, `cls_loss`, `l1_loss`) so voi YOLOv8 (`box_loss`, `cls_loss`, `dfl_loss`).

In [ ]:
import csv
import json
import os


def _ep_kieu_so(gia_tri):
    """Ep chuoi trong file CSV ve so thuc. Tra ve None neu o do bi trong."""
    try:
        return float(gia_tri)
    except (TypeError, ValueError):
        return None


def _do_cot_map(ten_cac_cot):
    """Do ten cot chua mAP@50 va mAP@50-95 trong file results.csv."""
    cot_map50 = None
    cot_map5095 = None
    for ten_cot in ten_cac_cot:
        ten_thuong = ten_cot.lower()
        if 'map50-95' in ten_thuong:
            cot_map5095 = ten_cot
        elif 'map50' in ten_thuong:
            cot_map50 = ten_cot
    return cot_map50, cot_map5095


def xuat_lich_su_ultralytics(thu_muc_ket_qua, ten_model, duong_dan_luu,
                             epochs_du_kien=None, patience=None):
    """Doc results.csv cua Ultralytics roi ghi ra file JSON lich su huan luyen.

    Moi epoch lay 5 truong: epoch_id, train_loss, val_loss, mAP_50, mAP_50_95.
    Loss tong = cong don tat ca cac cot co chu 'loss' (box + cls + dfl, hoac giou + cls + l1).
    """
    duong_dan_csv = os.path.join(thu_muc_ket_qua, 'results.csv')
    if not os.path.exists(duong_dan_csv):
        raise FileNotFoundError(f"Khong thay {duong_dan_csv}. Model da train xong chua?")

    lich_su = []
    with open(duong_dan_csv, 'r', encoding='utf-8') as f:
        bo_doc = csv.DictReader(f)
        ten_cac_cot = [c.strip() for c in bo_doc.fieldnames]
        cot_map50, cot_map5095 = _do_cot_map(ten_cac_cot)

        for so_thu_tu, dong in enumerate(bo_doc, start=1):
            dong = {k.strip(): v for k, v in dong.items() if k}

            # Cong don cac thanh phan loss cua tap train va tap val
            tong_loss_train = 0.0
            tong_loss_val = 0.0
            for ten_cot, gia_tri in dong.items():
                so = _ep_kieu_so(gia_tri)
                if so is None or 'loss' not in ten_cot.lower():
                    continue
                if ten_cot.lower().startswith('train'):
                    tong_loss_train += so
                elif ten_cot.lower().startswith('val'):
                    tong_loss_val += so

            epoch_id = _ep_kieu_so(dong.get('epoch')) or so_thu_tu

            lich_su.append({
                'epoch_id': int(epoch_id),
                'train_loss': round(tong_loss_train, 6),
                'val_loss': round(tong_loss_val, 6),
                'mAP_50': round(_ep_kieu_so(dong.get(cot_map50)) or 0.0, 6),
                'mAP_50_95': round(_ep_kieu_so(dong.get(cot_map5095)) or 0.0, 6),
            })

    ket_qua = {
        'model': ten_model,
        'epochs_du_kien': epochs_du_kien,
        'epochs_thuc_te': len(lich_su),
        'early_stopping_patience': patience,
        'nguon_du_lieu': duong_dan_csv,
        'history': lich_su,
    }

    os.makedirs(os.path.dirname(duong_dan_luu) or '.', exist_ok=True)
    with open(duong_dan_luu, 'w', encoding='utf-8') as f:
        json.dump(ket_qua, f, ensure_ascii=False, indent=2)

    print(f"Da ghi lich su huan luyen: {duong_dan_luu}")
    print(f"  Du kien {epochs_du_kien} epochs, thuc te chay {len(lich_su)} epochs.")
    if epochs_du_kien and len(lich_su) < epochs_du_kien:
        print("  => Early Stopping da kich hoat, model dung som vi khong con cai thien.")
    return ket_qua

## Cell 7: Sinh file `rtdetr_training_history.json`

In [ ]:
THU_MUC_KET_QUA = f'{SAVE_DIR}/zalo_traffic/rtdetr_v3'
DUONG_DAN_JSON = f'{SAVE_DIR}/rtdetr_training_history.json'

lich_su_rtdetr = xuat_lich_su_ultralytics(
    thu_muc_ket_qua=THU_MUC_KET_QUA,
    ten_model='RT-DETR-L',
    duong_dan_luu=DUONG_DAN_JSON,
    epochs_du_kien=NUM_EPOCHS,
    patience=PATIENCE,
)

for moc in lich_su_rtdetr['history'][:3]:
    print(moc)

## Cell 8: Ve Learning Curve

In [ ]:
import matplotlib.pyplot as plt


def ve_learning_curve(ket_qua_lich_su, duong_dan_luu_anh):
    """Ve 2 do thi canh nhau: duong cong Loss va duong cong mAP theo tung epoch."""
    lich_su = ket_qua_lich_su['history']
    cac_epoch = [m['epoch_id'] for m in lich_su]

    fig, (truc_trai, truc_phai) = plt.subplots(1, 2, figsize=(14, 5))

    # Do thi 1: Loss - dung de phat hien Overfitting (val_loss quay dau di len)
    truc_trai.plot(cac_epoch, [m['train_loss'] for m in lich_su], label='Train Loss')
    truc_trai.plot(cac_epoch, [m['val_loss'] for m in lich_su], label='Val Loss')
    truc_trai.set_xlabel('Epoch')
    truc_trai.set_ylabel('Loss')
    truc_trai.set_title(f"Learning Curve - {ket_qua_lich_su['model']}")
    truc_trai.legend()
    truc_trai.grid(alpha=0.3)

    # Do thi 2: mAP - dung de xac nhan diem hoi tu that su
    truc_phai.plot(cac_epoch, [m['mAP_50'] for m in lich_su], label='mAP@50')
    truc_phai.plot(cac_epoch, [m['mAP_50_95'] for m in lich_su], label='mAP@50-95')
    truc_phai.set_xlabel('Epoch')
    truc_phai.set_ylabel('mAP')
    truc_phai.set_title('Duong cong do chinh xac')
    truc_phai.legend()
    truc_phai.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(duong_dan_luu_anh, dpi=130)
    plt.show()
    print(f"Da luu bieu do: {duong_dan_luu_anh}")

In [ ]:
ve_learning_curve(lich_su_rtdetr, f'{SAVE_DIR}/rtdetr_learning_curve.png')

## Cell 9: Kiem tra trong so da nam an toan tren Drive

In [ ]:
import os

duong_dan_best = f'{THU_MUC_KET_QUA}/weights/best.pt'
if os.path.exists(duong_dan_best):
    dung_luong = os.path.getsize(duong_dan_best) / (1024 * 1024)
    print(f"OK - Trong so da luu tren Drive: {duong_dan_best} ({dung_luong:.1f} MB)")
    print("Tai file nay ve may, doi ten thanh 3_RTDETR_Large_Transformer.pt roi day len Kaggle Dataset.")
else:
    print("CANH BAO: Chua thay file best.pt. Kiem tra lai buoc huan luyen.")